In [ ]:
# 1) IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from src.geo import add_geo_features

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# 2) LOAD DATA

df = pd.read_csv("data/raw/nekretnine_dataset.csv")

df.head()

In [ ]:
# 3) BASIC INSPECTION

print(df.shape)
df.info()
df.describe()

In [ ]:
# 4) BASIC CLEANING

# square footage cleanup
df["Square_footage"] = (
    df["Square_footage"]
    .str.replace(" m²", "", regex=False)
    .astype(float)
)

df.head()

In [ ]:
# 5) MISSING VALUES

df.isnull().sum()

# remove missing price
df = df.dropna(subset=["Price_EUR"])

# remove missing square footage
df = df.dropna(subset=["Square_footage"])

# fill missing state values
df["State"] = df["State"].fillna("Standardna gradnja")

# fill missing heating values
df["Heating"] = df["Heating"].fillna("Centralno grejanje")

# fill missing room numbers using median m2 per room
median_m2_per_room = (df["Square_footage"] / df["Number_of_rooms"]).median()
print("Median m2 per room:", median_m2_per_room)

df.loc[df["Number_of_rooms"].isnull(), "Number_of_rooms"] = (
    df["Square_footage"] / median_m2_per_room
).round()

In [ ]:
# 6) OUTLIER CHECK

plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Price_EUR"])
plt.title("Price Outliers")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Square_footage"])
plt.title("Square Footage Outliers")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Number_of_rooms"])
plt.title("Room Count Outliers")
plt.show()

df.sort_values("Price_EUR", ascending=False).head(10)

In [ ]:
# 7) REMOVE EXTREME OUTLIERS

df = df[(df["Square_footage"] >= 15) & (df["Square_footage"] <= 400)]
df = df[(df["Number_of_rooms"] > 0) & (df["Number_of_rooms"] <= 10)]

min_price = 20000
max_price = 2000000
df = df[(df["Price_EUR"] >= min_price) & (df["Price_EUR"] <= max_price)]

# create price per m2 feature
df["price_per_m2"] = df["Price_EUR"] / df["Square_footage"]

# drop extreme apartments
df = df[df["price_per_m2"] <= 11000]

# reduce impossible room counts for tiny apartments
df.loc[df["Square_footage"] < 70, "Number_of_rooms"] = (
    df.loc[df["Square_footage"] < 70, "Number_of_rooms"].clip(upper=4)
)

print(df.describe())

In [ ]:
# 8) DISTRIBUTIONS

plt.figure(figsize=(14, 6))
sns.histplot(df["Price_EUR"], bins=50, kde=True)
plt.title("Distribution of Price")
plt.show()

plt.figure(figsize=(14, 6))
sns.histplot(df["price_per_m2"], bins=50, kde=True)
plt.title("Distribution of price_per_m2")
plt.show()

In [ ]:
# 9) CORRELATION HEATMAP

plt.figure(figsize=(10, 8))

corr = df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap", fontsize=16)
plt.show()

In [ ]:
# 10) BASIC RELATIONSHIPS

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x="Square_footage",
    y="Price_EUR",
    data=df,
    alpha=0.5
)
plt.title("Price vs Square Footage", fontsize=16)
plt.xlabel("Square Footage (m²)")
plt.ylabel("Price (EUR)")
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(
    df["price_per_m2"],
    bins=40,
    kde=True
)
plt.title("Distribution of Price per m²", fontsize=16)
plt.xlabel("Price per m² (EUR)")
plt.show()

In [ ]:
# 11) UNIQUE VALUES CHECK

print(df["State"].unique())
print(df["Heating"].unique())
print(df["Street"].unique())

In [ ]:
# 12) CATEGORICAL FEATURE ENGINEERING

state_map = {
    "Novogradnja": "New",
    "U izgradnji": "New",
    "Završena izgradnja": "New",
    "U pripremi": "New",
    "Standardna gradnja": "Standard",
    "Izvorno stanje": "Needs_renovation",
    "Kompletna rekonstrukcija": "Renovated",
    "Delimična rekonstrukcija": "Renovated",
    "Lux": "Luxury"
}

df["State"] = df["State"].map(state_map)
df["State"] = df["State"].fillna("Unknown")

heating_types = [
    "Centralno grejanje",
    "Etažno grejanje na gas",
    "Etažno grejanje na struju",
    "Etažno grejanje na čvrsto gorivo",
    "Podno grejanje",
    "Toplotna pumpa",
    "TA peć",
    "Kamin",
    "Peć na drva/ugalj",
    "Klima uređaj"
]

for heating in heating_types:
    df[f"heating_{heating}"] = df["Heating"].str.contains(heating, na=False).astype(int)

df.drop(columns=["Heating"], inplace=True)

In [ ]:
# 13) GEO FEATURES

df = add_geo_features(
    df,
    location_col="Street",
    cache_path=Path("geo_cache.csv"),
    user_agent="belgrade_real_estate_model",
    city_hint="Belgrade",
    country_hint="Serbia",
    delay_seconds=1,
    center_lat=44.8167,
    center_lon=20.4600
)

# fix broken coordinates
df = df[
    (df["lat"].between(44.6, 45.0)) &
    (df["lon"].between(20.2, 20.7))
]

# drop helper column if present
if "geo_missing" in df.columns:
    df = df.drop(columns=["geo_missing"])

In [ ]:
# 14) GEO VISUALS

sns.scatterplot(x="dist_to_center_km", y="Price_EUR", data=df)
plt.title("Price vs Distance to Center")
plt.show()

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="State", y="price_per_m2")
plt.xticks(rotation=45)
plt.title("Price per m² by Apartment State")
plt.show()

plt.figure(figsize=(10, 8))
plt.scatter(
    df["lon"],
    df["lat"],
    c=df["price_per_m2"],
    cmap="viridis",
    alpha=0.7
)
plt.colorbar(label="Price per m² (EUR)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Apartment Price per m² across Belgrade")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="dist_to_center_km",
    y="price_per_m2",
    alpha=0.6
)
sns.regplot(
    data=df,
    x="dist_to_center_km",
    y="price_per_m2",
    scatter=False,
    color="red"
)
plt.title("Price per m² vs Distance to Belgrade Center")
plt.xlabel("Distance to Center (km)")
plt.ylabel("Price per m² (EUR)")
plt.show()

plt.figure(figsize=(10, 8))
plt.hexbin(
    df["lon"],
    df["lat"],
    C=df["price_per_m2"],
    gridsize=40,
    cmap="plasma"
)
plt.colorbar(label="Average price per m²")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic Heatmap of Apartment Prices")
plt.show()

In [ ]:
# 15) ZONES

print(df.columns)

df["zone"] = pd.cut(
    df["dist_to_center_km"],
    bins=[0, 3, 6, 10, 25],
    labels=["Center", "Inner city", "Outer city", "Suburbs"]
)

heating_cols = [c for c in df.columns if c.startswith("heating_")]

heating_zone = df.groupby("zone")[heating_cols].sum()

heating_zone.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="viridis"
)
plt.title("Heating Types by Distance Zone")
plt.ylabel("Number of Apartments")
plt.show()

In [ ]:
# 16) MORE GEO ANALYSIS

plt.figure(figsize=(10, 8))
plt.scatter(
    df["lon"],
    df["lat"],
    c=df["Optical_internet"],
    cmap="coolwarm",
    alpha=0.6
)
plt.colorbar(label="Optical Internet")
plt.title("Geographic Distribution of Optical Internet")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

print(df[["dist_to_center_km", "Optical_internet"]].corr())

parking_zone = df.groupby("zone")["Parking"].mean()

plt.figure(figsize=(9, 6))
parking_zone.plot(kind="bar", color="#2E86AB")
plt.title("Parking Availability by Distance Zone")
plt.ylabel("Share of Apartments with Parking")
plt.xlabel("City Zone")
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df,
    x="zone",
    y="price_per_m2",
    palette="coolwarm"
)
plt.title("Price per m² by Distance Zone")
plt.ylabel("Price per m² (EUR)")
plt.xlabel("City Zone")
plt.show()

lift_zone = df.groupby("zone")["Lift"].mean()

plt.figure(figsize=(9, 6))
lift_zone.plot(kind="bar", color="#F18F01")
plt.title("Lift Availability by Distance Zone")
plt.ylabel("Share of Apartments with Lift")
plt.xlabel("City Zone")
plt.show()

plt.figure(figsize=(10, 10))
sns.scatterplot(
    data=df,
    x="lon",
    y="lat",
    hue="price_per_m2",
    palette="coolwarm",
    alpha=0.7
)
plt.title("Belgrade Apartment Prices per m² (Geographical Distribution)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Price per m²", bbox_to_anchor=(1.05, 1))
plt.show()

In [ ]:
# 17) FLOOR CLEANUP

print(df["Floor"].unique())

floor_mapping = {
    "Suteren": -1,
    "Prizemlje": 0,
    "Visoko prizemlje": 0.5
}

df["Floor"] = df["Floor"].replace(floor_mapping)
df["Floor"] = pd.to_numeric(df["Floor"], errors="coerce")

In [ ]:
# 17.1) FEATURE ENGINEERING (za linear model)

# log transform feature-a (pomaže linearnosti)
df["log_square_footage"] = np.log1p(df["Square_footage"])

# odnos soba i kvadrature (gustina)
df["rooms_per_m2"] = df["Number_of_rooms"] / df["Square_footage"]

# interaction feature (veličina × lokacija)
df["size_x_distance"] = df["Square_footage"] * df["dist_to_center_km"]

# kvadrat distance (hvata nelinearnost)
df["dist_squared"] = df["dist_to_center_km"] ** 2

# opcionalno: cena po sobi (ako želiš agresivnije)
# df["price_per_room"] = df["Price_EUR"] / df["Number_of_rooms"]

print("Feature engineering done.")

In [ ]:
# 18) MODELING SETUP 
#    RidgeCV baseline + Decision Tree baseline
#    Random Forest optimized + XGBoost optimized

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

df_model = df.copy()

drop_cols = ["Price_EUR", "Street", "zone", "geo_status", "price_per_m2"]
drop_cols = [c for c in drop_cols if c in df_model.columns]

X = df_model.drop(columns=drop_cols).copy()
y = np.log1p(df_model["Price_EUR"].astype(float))

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()

binary_cols = []
for col in numerical_cols:
    vals = set(X[col].dropna().unique())
    if vals.issubset({0, 1}) or vals.issubset({False, True}):
        binary_cols.append(col)

continuous_cols = [c for c in numerical_cols if c not in binary_cols]

if "State" in X.columns:
    X["State"] = X["State"].fillna("Unknown").astype(str)

print("Categorical cols:", categorical_cols)
print("Continuous cols:", continuous_cols)
print("Binary cols:", binary_cols)

In [ ]:
# 19) TRAIN / TEST SPLIT + HELPERS

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

def inverse_target(arr):
    return np.expm1(arr)

def evaluate_model(model, X_test, y_test):
    y_pred_log = model.predict(X_test)

    y_true = inverse_target(y_test)
    y_pred = inverse_target(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2

try:
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse=False)

In [ ]:
# 20) PREPROCESSORS

preprocessor_linear = ColumnTransformer(
    transformers=[
        ("cont", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), continuous_cols),

        ("bin", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ]), binary_cols),

        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe_dense)
        ]), categorical_cols)
    ],
    remainder="drop"
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("cont", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), continuous_cols),

        ("bin", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ]), binary_cols),

        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe_dense)
        ]), categorical_cols)
    ],
    remainder="drop"
)

results = []

In [ ]:
# 21) BASELINE 1: RIDGECV

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_linear),
    ("model", RidgeCV(alphas=np.logspace(-4, 4, 40), cv=5))
])

ridge_pipeline.fit(X_train, y_train)
mae_ridge, rmse_ridge, r2_ridge = evaluate_model(ridge_pipeline, X_test, y_test)

results.append({
    "Model": "RidgeCV (Linear)",
    "MAE": mae_ridge,
    "RMSE": rmse_ridge,
    "R2": r2_ridge
})

print(f"RidgeCV -> MAE: {mae_ridge:.2f} EUR | RMSE: {rmse_ridge:.2f} EUR | R2: {r2_ridge:.4f}")

In [ ]:
# 22) BASELINE 2: DECISION TREE

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", DecisionTreeRegressor(max_depth=8,
        min_samples_leaf=5,
        min_samples_split=10,
        random_state=42))
])

dt_pipeline.fit(X_train, y_train)
mae_dt, rmse_dt, r2_dt = evaluate_model(dt_pipeline, X_test, y_test)

results.append({
    "Model": "Decision Tree",
    "MAE": mae_dt,
    "RMSE": rmse_dt,
    "R2": r2_dt
})

print(f"Decision Tree -> MAE: {mae_dt:.2f} EUR | RMSE: {rmse_dt:.2f} EUR | R2: {r2_dt:.4f}")

In [ ]:
# 23) OPTIMIZED RANDOM FOREST

rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_param_dist = {
    "model__n_estimators": [200, 300, 500, 700, 1000],
    "model__max_depth": [None, 10, 20, 30, 40, 50],
    "model__min_samples_split": [2, 5, 10, 15],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.7, 1.0],
    "model__bootstrap": [True, False]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_

mae_rf_opt, rmse_rf_opt, r2_rf_opt = evaluate_model(rf_best, X_test, y_test)

results.append({
    "Model": "Random Forest Optimized",
    "MAE": mae_rf_opt,
    "RMSE": rmse_rf_opt,
    "R2": r2_rf_opt
})

print("Best RF params:", rf_search.best_params_)
print(f"Random Forest Optimized -> MAE: {mae_rf_opt:.2f} EUR | RMSE: {rmse_rf_opt:.2f} EUR | R2: {r2_rf_opt:.4f}")

In [ ]:
# 24) OPTIMIZED XGBOOST

xgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", XGBRegressor(
        random_state=42,
        tree_method="hist"
    ))
])

xgb_param_dist = {
    "model__n_estimators": [300, 500, 800, 1200, 1600],
    "model__max_depth": [3, 4, 5, 6, 7, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__gamma": [0, 0.1, 0.2, 0.5],
    "model__reg_alpha": [0, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda": [0.5, 1.0, 1.5, 2.0, 5.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_

mae_xgb_opt, rmse_xgb_opt, r2_xgb_opt = evaluate_model(xgb_best, X_test, y_test)

results.append({
    "Model": "XGBoost Optimized",
    "MAE": mae_xgb_opt,
    "RMSE": rmse_xgb_opt,
    "R2": r2_xgb_opt
})

print("Best XGB params:", xgb_search.best_params_)
print(f"XGBoost Optimized -> MAE: {mae_xgb_opt:.2f} EUR | RMSE: {rmse_xgb_opt:.2f} EUR | R2: {r2_xgb_opt:.4f}")

In [ ]:
# 24.5) FEATURE IMPORTANCE / COEFFICIENTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def get_feature_names_from_preprocessor(preprocessor):
    """Return transformed feature names from a fitted ColumnTransformer."""
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        # fallback if sklearn version is older
        return None

def plot_top_features(feature_names, importances, title, top_n=20):
    df_imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    df_imp["Importance_abs"] = df_imp["Importance"].abs()

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=df_imp.head(top_n),
        x="Importance_abs",
        y="Feature"
    )
    plt.title(title)
    plt.xlabel("Absolute importance / coefficient magnitude")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return df_imp

# Ridge coefficients

ridge_feature_names = get_feature_names_from_preprocessor(
    ridge_pipeline.named_steps["preprocessor"]
)

if ridge_feature_names is not None:
    ridge_coefs = ridge_pipeline.named_steps["model"].coef_

    ridge_imp = pd.DataFrame({
        "Feature": ridge_feature_names,
        "Coefficient": ridge_coefs,
        "Abs_Coefficient": np.abs(ridge_coefs)
    }).sort_values("Abs_Coefficient", ascending=False)

    print("\nTop RidgeCV coefficients:")
    print(ridge_imp.head(20))

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=ridge_imp.head(20),
        x="Abs_Coefficient",
        y="Feature"
    )
    plt.title("Top 20 RidgeCV Features by Absolute Coefficient")
    plt.xlabel("Absolute coefficient")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

# Decision Tree importance

dt_feature_names = get_feature_names_from_preprocessor(
    dt_pipeline.named_steps["preprocessor"]
)

if dt_feature_names is not None:
    dt_importances = dt_pipeline.named_steps["model"].feature_importances_

    dt_imp = pd.DataFrame({
        "Feature": dt_feature_names,
        "Importance": dt_importances
    }).sort_values("Importance", ascending=False)

    print("\nTop Decision Tree importances:")
    print(dt_imp.head(20))

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=dt_imp.head(20),
        x="Importance",
        y="Feature"
    )
    plt.title("Top 20 Decision Tree Features")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

# Random Forest importance

rf_feature_names = get_feature_names_from_preprocessor(
    rf_best.named_steps["preprocessor"]
)

if rf_feature_names is not None:
    rf_importances = rf_best.named_steps["model"].feature_importances_

    rf_imp = pd.DataFrame({
        "Feature": rf_feature_names,
        "Importance": rf_importances
    }).sort_values("Importance", ascending=False)

    print("\nTop Random Forest importances:")
    print(rf_imp.head(20))

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=rf_imp.head(20),
        x="Importance",
        y="Feature"
    )
    plt.title("Top 20 Random Forest Features")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

# XGBoost importance

xgb_feature_names = get_feature_names_from_preprocessor(
    xgb_best.named_steps["preprocessor"]
)

if xgb_feature_names is not None:
    xgb_importances = xgb_best.named_steps["model"].feature_importances_

    xgb_imp = pd.DataFrame({
        "Feature": xgb_feature_names,
        "Importance": xgb_importances
    }).sort_values("Importance", ascending=False)

    print("\nTop XGBoost importances:")
    print(xgb_imp.head(20))

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=xgb_imp.head(20),
        x="Importance",
        y="Feature"
    )
    plt.title("Top 20 XGBoost Features")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

In [ ]:
# 25) FINAL RESULTS TABLE + VISUALIZATION

results_df = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
print(results_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by RMSE")
plt.ylabel("RMSE (EUR)")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="Model", y="MAE")
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by MAE")
plt.ylabel("MAE (EUR)")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

In [ ]:
# 26) BEST MODEL DIAGNOSTICS

best_model = xgb_best

y_pred_best_log = best_model.predict(X_test)
y_pred_best = np.expm1(y_pred_best_log)
y_true_best = np.expm1(y_test)

plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_true_best, y=y_pred_best, alpha=0.5)
plt.plot([y_true_best.min(), y_true_best.max()],
         [y_true_best.min(), y_true_best.max()],
         color="red", linestyle="--")
plt.title("Actual vs Predicted Prices")
plt.xlabel("Actual Price (EUR)")
plt.ylabel("Predicted Price (EUR)")
plt.tight_layout()
plt.show()

residuals = y_true_best - y_pred_best

plt.figure(figsize=(10, 6))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residuals (EUR)")
plt.tight_layout()
plt.show()